In [ ]:
!pip install geopandas rioxarray --quiet

In [1]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import rioxarray as rxr
import pandas as pd

# ==========================================
# 1. Parameter Utama & Koordinat Indonesia
# ==========================================
cwd = os.getcwd()
tahun_awal = 2025
tahun_akhir = 2026

# Koordinat Ekstrem Indonesia (Bounding Box)
# Format: Bujur Barat (minx), Lintang Selatan (miny), Bujur Timur (maxx), Lintang Utara (maxy)
minx, miny, maxx, maxy = 95.0, -11.0, 141.0, 6.0

print(f"Membaca batas koordinat wilayah Indonesia...\nMin Lon: {minx}, Min Lat: {miny}\nMax Lon: {maxx}, Max Lat: {maxy}")

# Folder induk untuk semua data CHIRPS
folder_induk = os.path.join(cwd, "data", "CHIRPS_Indonesia")
os.makedirs(folder_induk, exist_ok=True)

print(f"Folder induk output: {folder_induk}")

# ==========================================
# 2. Loop Unduh, Potong, dan Konversi
# ==========================================
for tahun in range(tahun_awal, tahun_akhir + 1):
    url_sumber = f"https://data.chc.ucsb.edu/products/CHIRPS/v3.0/daily/final/rnl/{tahun}/"
    folder_tahun = os.path.join(folder_induk, str(tahun))
    os.makedirs(folder_tahun, exist_ok=True)

    print(f"\n=== Mengakses Tahun {tahun} ===")
    print(f"Direktori web: {url_sumber}")
    
    try:
        response = requests.get(url_sumber, timeout=60)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
    except Exception as e:
        print(f"Gagal membaca direktori tahun {tahun}: {e}")
        continue

    # 3. Cari semua link (tag <a>)
    for link in soup.find_all("a"):
        nama_file = link.get("href")

        if not nama_file or nama_file in ("../", "/"):
            continue

        # Filter: Hanya unduh file dengan ekstensi tertentu
        if nama_file.endswith(".tif") or nama_file.endswith(".tif.gz"):
            link_download = urljoin(url_sumber, nama_file)
            
            # --- EKSTRAK TANGGAL DI AWAL ---
            bagian = nama_file.split('.')
            thn, bln, hr = bagian[3], bagian[4], bagian[5]
            
            # --- BUAT SUB-FOLDER BULAN ---
            folder_bulan = os.path.join(folder_tahun, f"{thn}_{bln}")
            os.makedirs(folder_bulan, exist_ok=True)
            
            # --- TENTUKAN NAMA & LOKASI OUTPUT .nc ---
            nama_file_output = nama_file.replace('.gz', '').replace('.tif', '.nc')
            path_simpan = os.path.join(folder_bulan, nama_file_output)

            # Skip jika file sudah pernah diunduh & dipotong
            if not os.path.exists(path_simpan):
                print(f"Memproses: {thn}-{bln}-{hr} -> Folder {thn}_{bln}/ ... ", end="")
                
                # Path file sementara
                temp_path = os.path.join(cwd, f"temp_global_{nama_file}")
                
                try:
                    # A. UNDUH FILE GLOBAL SEMENTARA
                    with requests.get(link_download, stream=True, timeout=120) as r:
                        r.raise_for_status()
                        with open(temp_path, "wb") as f:
                            for chunk in r.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                    
                    # B. POTONG FILE MENGGUNAKAN KOORDINAT (CLIP_BOX)
                    da = rxr.open_rasterio(temp_path, masked=True)
                    
                    # Kita menggunakan clip_box() sebagai pengganti clip()
                    da_clipped = da.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)
                    
                    # C. INJEKSI WAKTU & KONVERSI KE NETCDF
                    tanggal = pd.to_datetime(f"{thn}-{bln}-{hr}")
                    
                    # Rapikan variabel dan dimensi
                    da_clipped = da_clipped.squeeze("band", drop=True) if "band" in da_clipped.dims else da_clipped
                    da_clipped.name = "precipitation"
                    da_clipped = da_clipped.assign_coords(time=tanggal)
                    da_clipped = da_clipped.expand_dims("time")
                    
                    # Simpan hasil potongan
                    da_clipped.to_netcdf(path_simpan)
                    
                    # D. BERSIHKAN MEMORI & HAPUS FILE SEMENTARA
                    da.close()
                    da_clipped.close()
                    os.remove(temp_path)
                    
                    print("Selesai ✓")
                    
                except Exception as e:
                    print(f" Gagal ({e})")
                    if os.path.exists(temp_path):
                        os.remove(temp_path)
            else:
                pass 

print("\n🎉 Proses unduhan dan pemotongan berdasarkan kotak koordinat selesai!")

Membaca batas koordinat wilayah Indonesia...
Min Lon: 95.0, Min Lat: -11.0
Max Lon: 141.0, Max Lat: 6.0
Folder induk output: d:\Github\Projek_Downscale\data\CHIRPS_Indonesia

=== Mengakses Tahun 2025 ===
Direktori web: https://data.chc.ucsb.edu/products/CHIRPS/v3.0/daily/final/rnl/2025/
Memproses: 2025-01-01 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-02 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-03 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-04 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-05 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-06 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-07 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-08 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-09 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-10 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-11 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-12 -> Folder 2025_01/ ... Selesai ✓
Memproses: 2025-01-13 -> Folder 2025_01/ ... Selesai